In [6]:
import casadi as cs
import numpy as np
from scipy.optimize import minimize  # (not used in the RL version)
import matplotlib.pyplot as plt
import time
from sklearn.metrics import mean_absolute_error, mean_squared_error

# -----------------------------
# Quadrotor dynamics and helpers
# -----------------------------
class Quadrotor:
    def __init__(self, quad_params):
        self.quad = quad_params
        self.x = cs.MX.sym('x', 13)  # State: 13-dimensional vector
        self.u = cs.MX.sym('u', 4)   # Control inputs: 4-dimensional

    def quad_dynamics(self):
        # x_dot = [p_dynamics; q_dynamics; v_dynamics; w_dynamics]
        x_dot = cs.vertcat(self.p_dynamics(),
                           self.q_dynamics(),
                           self.v_dynamics(),
                           self.w_dynamics())
        return cs.Function('x_dot', [self.x, self.u], [x_dot], ['x', 'u'], ['x_dot'])

    def p_dynamics(self):
        return self.x[6:9]  # Position derivative (velocity)

    def q_dynamics(self):
        # quaternion derivative
        return 0.5 * cs.mtimes(skew_symmetric(self.x[9:13]), self.x[3:7])

    def v_dynamics(self):
        # Thrust force and gravity combine to yield acceleration
        f_thrust = self.u * self.quad['max_thrust']
        g = cs.vertcat(0.0, 0.0, 9.81)
        a_thrust = cs.vertcat(0.0, 0.0, cs.sum1(f_thrust)) / self.quad['mass']
        v_dynamics = v_dot_q(a_thrust, self.x[3:7]) - g
        return v_dynamics

    def w_dynamics(self):
        f_thrust = self.u * self.quad['max_thrust']
        y_f = cs.MX(self.quad['y_f'])
        x_f = cs.MX(self.quad['x_f'])
        c_f = cs.MX(self.quad['z_l_tau'])
        return cs.vertcat(
            (cs.mtimes(f_thrust.T, y_f) + (self.quad['J'][1] - self.quad['J'][2]) * self.x[11] * self.x[12]) / self.quad['J'][0],
            (-cs.mtimes(f_thrust.T, x_f) + (self.quad['J'][2] - self.quad['J'][0]) * self.x[12] * self.x[10]) / self.quad['J'][1],
            (cs.mtimes(f_thrust.T, c_f) + (self.quad['J'][0] - self.quad['J'][1]) * self.x[10] * self.x[11]) / self.quad['J'][2]
        )

# Helper functions
def v_dot_q(v, q):
    rot_mat = q_to_rot_mat(q)
    if isinstance(q, np.ndarray):
        return rot_mat.dot(v)
    return cs.mtimes(rot_mat, v)

def q_to_rot_mat(q):
    # q = [qw, qx, qy, qz]
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    if isinstance(q, np.ndarray):
        return np.array([
            [1 - 2*(qy**2 + qz**2), 2*(qx*qy - qw*qz), 2*(qx*qz + qw*qy)],
            [2*(qx*qy + qw*qz), 1 - 2*(qx**2 + qz**2), 2*(qy*qz - qw*qx)],
            [2*(qx*qz - qw*qy), 2*(qy*qz + qw*qx), 1 - 2*(qx**2 + qy**2)]
        ])
    return cs.vertcat(
        cs.horzcat(1 - 2*(qy**2 + qz**2), 2*(qx*qy - qw*qz), 2*(qx*qz + qw*qy)),
        cs.horzcat(2*(qx*qy + qw*qz), 1 - 2*(qx**2 + qz**2), 2*(qy*qz - qw*qx)),
        cs.horzcat(2*(qx*qz - qw*qy), 2*(qy*qz + qw*qx), 1 - 2*(qx**2 + qy**2))
    )

def skew_symmetric(v):
    # Works for both NumPy arrays and CasADi MX variables.
    if isinstance(v, np.ndarray):
        return np.array([[0, -v[0], -v[1], -v[2]],
                         [v[0], 0, v[2], -v[1]],
                         [v[1], -v[2], 0, v[0]],
                         [v[2], v[1], -v[0], 0]])
    return cs.vertcat(
        cs.horzcat(0, -v[0], -v[1], -v[2]),
        cs.horzcat(v[0], 0, v[2], -v[1]),
        cs.horzcat(v[1], -v[2], 0, v[0]),
        cs.horzcat(v[2], v[1], -v[0], 0)
    )

# -----------------------------
# Quadrotor parameters and data
# -----------------------------
quad_params = {
    'mass': 1.0,
    'max_thrust': 5.0,
    'y_f': np.array([0.1, -0.1, -0.1, 0.1]),  # Four elements
    'x_f': np.array([0.1, 0.1, -0.1, -0.1]),    # Four elements
    'z_l_tau': np.array([-0.05, 0.05, -0.05, 0.05]),  # Four elements
    'J': np.array([0.01, 0.01, 0.02])
}

# Load provided data (trajectories, optimal controls, cost matrices)
trajectories = np.load('fractional_system_trajectories.npy')
U_optimal = np.load('optimal_control_U.npy')
Q_matrices = np.load('LQR_Q.npy')
R_matrices = np.load('LQR_R.npy')

# -----------------------------
# Simulation using a linear policy (model-based RL)
# -----------------------------
def simulate_policy_quad(initial_state, K, Q, R, params, dt, steps, u_bounds):
    """
    Simulate the quadrotor using a linear policy u = -K x over a given number of steps.
    
    Args:
        initial_state: 13-dimensional numpy array.
        K: Control gain matrix of shape (4, 13).
        Q: State cost matrix (13x13).
        R: Control cost matrix (4x4).
        params: Quadrotor parameters.
        dt: Time step.
        steps: Number of simulation steps.
        u_bounds: Tuple (lower_bound, upper_bound) applied elementwise to u.
        
    Returns:
        trajectory: Array of states (steps+1 x 13).
        controls: Array of controls (steps x 4).
        total_cost: Cumulative cost (scalar) computed as sum(state.T Q state + u.T R u) plus terminal cost.
    """
    state = initial_state.copy()
    trajectory = [state.copy()]
    controls = []
    total_cost = 0

    # Create the quadrotor dynamics function once.
    quad = Quadrotor(params)
    quad_dyn = quad.quad_dynamics()

    for i in range(steps):
        # Compute control using the linear policy (u = -K x)
        # (.item() is not needed here because u is a vector)
        u = -K @ state  # shape (4,)
        u = np.clip(u, u_bounds[0], u_bounds[1])
        controls.append(u)
        
        # Increment cost: quadratic cost for state and control
        total_cost += state.T @ Q @ state + u.T @ R @ u

        # Compute state derivative using the CasADi dynamics function
        x_dot = np.array(quad_dyn(x=cs.DM(state), u=cs.DM(u))['x_dot']).flatten()
        state = state + dt * x_dot
        trajectory.append(state.copy())
        
    total_cost += state.T @ Q @ state  # terminal cost
    return np.array(trajectory), np.array(controls), total_cost

# -----------------------------
# Policy training via random search
# -----------------------------
def train_policy_model_based_quad(initial_state, params, Q, R, dt, steps, u_bounds, iterations=100, alpha=0.01):
    """
    Optimize the linear policy parameters K (shape 4x13) using random search.
    
    Args:
        initial_state: initial state (13-dimensional vector)
        params: Quadrotor parameters
        Q: State cost matrix (13x13)
        R: Control cost matrix (4x4)
        dt: Time step
        steps: Number of simulation steps for rollout
        u_bounds: Control input bounds (tuple)
        iterations: Number of random search iterations
        alpha: Standard deviation for random perturbation
        
    Returns:
        K_best: Optimized gain matrix (4x13)
        best_cost: The cost achieved by the best policy.
    """
    d = len(initial_state)  # 13
    K_best = np.zeros((4, d))  # initial guess: zero matrix
    # Evaluate initial guess
    _, _, best_cost = simulate_policy_quad(initial_state, K_best, Q, R, params, dt, steps, u_bounds)
    
    for _ in range(iterations):
        # Randomly perturb the current best gain matrix
        K_candidate = K_best + alpha * np.random.randn(4, d)
        _, _, cost_candidate = simulate_policy_quad(initial_state, K_candidate, Q, R, params, dt, steps, u_bounds)
        if cost_candidate < best_cost:
            best_cost = cost_candidate
            K_best = K_candidate
    return K_best, best_cost

# -----------------------------
# Main simulation loop (model-based RL version)
# -----------------------------
mses = []
maes = []
time_horizon = 8  # prediction horizon (if needed for comparison; not used in RL version)
u_bounds = (-1, 1) 
dt = 0.1           # time step
T = 1.6            # total simulation time (same as before)
N = int(T / dt)    # number of steps

start_time = time.time()

for i in range(2900, 2910):
    # Initial state from data
    x0 = trajectories[i, 0]

    # Cost matrices for this sample
    Q = Q_matrices[i]
    R = R_matrices[i]

    # Train the policy for the current initial state
    K, train_cost = train_policy_model_based_quad(x0, quad_params, Q, R, dt, N, u_bounds,
                                                   iterations=100, alpha=0.01)

    # Evaluate the learned policy using a full rollout
    traj, controls_rl, rollout_cost = simulate_policy_quad(x0, K, Q, R, quad_params, dt, N, u_bounds)
    
    # Compare the RL-generated control sequence to the optimal control sequence from MPC
    mae = mean_absolute_error(controls_rl, U_optimal[i])
    mse = mean_squared_error(controls_rl, U_optimal[i])
    print(f"Iteration {i}, MAE: {mae:.4f}, MSE: {mse:.4f}")
    maes.append(mae)
    mses.append(mse)

end_time = time.time()
total_runtime = end_time - start_time
print("Total runtime: {:.2f} seconds".format(total_runtime))

Iteration 2900, MAE: 0.5271, MSE: 0.5973
Iteration 2901, MAE: 0.3311, MSE: 0.2103
Iteration 2902, MAE: 0.6114, MSE: 0.5358
Iteration 2903, MAE: 0.5851, MSE: 0.5855
Iteration 2904, MAE: 0.6552, MSE: 0.4914
Iteration 2905, MAE: 0.6010, MSE: 0.4346
Iteration 2906, MAE: 0.8774, MSE: 1.1828
Iteration 2907, MAE: 0.2243, MSE: 0.0967
Iteration 2908, MAE: 0.8761, MSE: 0.8900
Iteration 2909, MAE: 0.7534, MSE: 0.6608
Total runtime: 1.57 seconds
